In [ ]:
import os
import requests
import pandas as pd
from statsbombpy import sb
import warnings

# Suppress the NoAuthWarning as intentionally using Open Data
warnings.filterwarnings("ignore", category=UserWarning, module="statsbombpy")

# Folder for offline data
os.makedirs("sb_offline_data", exist_ok=True)

# Loading seasons
comps = sb.competitions()
la_liga = comps[(comps.competition_name == "La Liga") & 
                (comps.season_name.isin(["2015/2016", "2016/2017", "2017/2018", "2018/2019", "2019/2020", "2020/2021"]))]
prem = comps[(comps.competition_name == "Premier League") & 
             (comps.season_name.isin(["2015/2016"]))]

match_ids = []

# Collecting match IDs for La Liga and Premier League
for _, row in pd.concat([la_liga, prem], ignore_index=True).iterrows():
    m = sb.matches(competition_id=row.competition_id, season_id=row.season_id)
    match_ids.extend(m.match_id.tolist())

# Fixed Bundesliga match IDs from the corrected source
url = "https://raw.githubusercontent.com/statsbomb/open-data/add-womens-euros-2025/data/matches/9/27.json"
response = requests.get(url)
bundesliga_matches = pd.DataFrame(response.json())
match_ids.extend(bundesliga_matches.match_id.tolist())

# Unique match IDs
match_ids = list(set(match_ids))
print(f"Number of unique matches: {len(match_ids)}")

# Download and save events for each match
for i, m_id in enumerate(match_ids):
    file_name = f"sb_offline_data/match_{m_id}.pkl"
    
    # Check if the file already exists to avoid re-downloading
    if not os.path.exists(file_name):
        try:
            events = sb.events(match_id=m_id)
            events.to_pickle(file_name)
        except Exception as e:
            print(f"Error downloading {m_id}: {e}")
            
    if (i + 1) % 50 == 0:
        print(f"Downloaded: {i + 1}/{len(match_ids)} matches processed.")
